This notebook creates a CSV file that serves as the backend dataset for the dashboard. The dataset contains one row for each combination of LSOA and crime type. To ensure that all LSOAs in England and Wales are represented, the dataset is initialized using the LSOA location information dataset, resulting in all 35,672 LSOAs being included, even if no crimes were recorded for a particular crime type.

The generated CSV contains the following attributes:
* **LSOA code** (*str*)
  The unique code identifying the LSOA. All 35,672 LSOAs in England and Wales are included.

* **LSOA name** (*str*)
  The name of the LSOA.

* **Crime type** (*str*)
  The crime category associated with the row.

* **last_month_count** (*float*)
  The number of crimes of the specified crime type recorded in the most recent month available in the dataset for the given LSOA.

* **last_year_monthly_avg** (*float*)
  The average monthly number of crimes of the specified crime type in the given LSOA over the previous 12 months.

* **max_monthly_crime_count** (*float*)
  The highest number of crimes of the specified crime type recorded in a single month for the given LSOA across the entire dataset.

* **lsoa_total_last_month** (*float*)
  The total number of crimes of all crime types recorded in the given LSOA during the most recent month available in the dataset.

* **last_month_percentage** (*float*)
  The percentage of all crimes in the LSOA during the most recent month that belong to the specified crime type.

* **last_year_percentage** (*float*)
  The percentage of all crimes in the LSOA during the previous 12 months that belong to the specified crime type.

* **police_force** (*str*)
  The police force area to which the LSOA is assigned. A small number of LSOAs intersect multiple police force boundaries. For consistency and to support resource allocation, each LSOA is assigned to a single police force.

* **hotspot_status** (*bool*)
  Indicates whether the LSOA is classified as a hotspot. This information is derived from the `(non)hotspot_dictionary.json` file generated by `identify_hotspots.py`.

* **all_time_monthly_avg** (*float*)
  The average monthly number of crimes of the specified crime type in the given LSOA across the entire dataset (3 years in our case).


Graphs I think would be useful to add on the dashboard:
-	A grouped bar chart that shows the distribution of crime types of a selected LSOA in percentage/counts for both the historical and the predicted data. It can help identify the main crime type and verify if that crime type is actually a lot or if it is just a 100% increase from 1->2
-	A line chart showing both historical and predicted crime counts, maybe try to overlap them that one line is in past and one in the future, so it is easier to compare (future line we already have). It helps to show trends.
-	Bar chart that shows the percentage of increase/decrease of the forecast for each crime type (and maybe total?) compared to the baseline average of last year
-	Maybe show the rank of the crime type of that police force (e.g. 3rd highest burglary LSOA within the force) An LSOA may show a high percentage increase in a crime type, but if it ranks relatively low compared to other LSOAs in the force, it may be less of a priority for resource allocation.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import geopandas as gpd
import json

In [2]:
# Reading the df_all_street
df_all_street = pd.read_csv("../data/all_street.csv")
print("All street data has been read")

All street data has been read


In [3]:
# Reading the information about LSOA's
base_path = "../data/LSOA_location_info"
shapefiles = glob.glob(os.path.join(base_path, "**/*.shp"), recursive=True)

lsoa_gdfs = [gpd.read_file(shp) for shp in shapefiles]
lsoa_locations = gpd.GeoDataFrame(
    pd.concat(lsoa_gdfs, ignore_index=True),
    crs=lsoa_gdfs[0].crs
)
lsoa_locations.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   LSOA21CD   35672 non-null  str     
 1   LSOA21NM   35672 non-null  str     
 2   LSOA21NMW  1917 non-null   str     
 3   BNG_E      35672 non-null  int64   
 4   BNG_N      35672 non-null  int64   
 5   LAT        35672 non-null  float64 
 6   LONG       35672 non-null  float64 
 7   GlobalID   35672 non-null  str     
 8   geometry   35672 non-null  geometry
dtypes: float64(2), geometry(1), int64(2), str(4)
memory usage: 4.6 MB


In [4]:
# Read information about force boundaries
kml_folder = "../data/force kmls"
kml_files = glob.glob(os.path.join(kml_folder, "**/*.kml"), recursive=True)

force_gdfs = []
for file in kml_files:
    gdf = gpd.read_file(file, driver="KML")
    gdf["force"] = os.path.basename(file).replace(".kml", "")
    force_gdfs.append(gdf)

forces = gpd.GeoDataFrame(pd.concat(force_gdfs, ignore_index=True))
forces = forces.to_crs(lsoa_locations.crs)
lsoa_locations = lsoa_locations.to_crs(forces.crs)
forces["geometry"] = forces["geometry"].buffer(0)

In [5]:
# Removing unnecessary information
if "Context" in df_all_street.columns:
    df_all_street = df_all_street.drop(columns=["Context"])
    df_all_street = df_all_street.drop(columns=["Longitude"])
    df_all_street = df_all_street.drop(columns=["Latitude"])
    df_all_street = df_all_street.drop(columns=["Location"])
    df_all_street = df_all_street.drop(columns=["Last outcome category"])

df_all_street = df_all_street.dropna(subset=["LSOA name", "LSOA code"])
df_all_street = df_all_street[df_all_street["Reported by"] != "British Transport Police"]

In [6]:
# Create a df that contains all the LSOA codes and names in England and Wales so none are missing
df_all_lsoas = (
    lsoa_locations[
        ["LSOA21CD", "LSOA21NM"]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "LSOA21CD": "LSOA code",
            "LSOA21NM": "LSOA name"
        }
    )
)
df_all_lsoas.info()
# Non-Null count is 35672 these are indeed all LSOA's of England and Wales

<class 'pandas.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   LSOA code  35672 non-null  str  
 1   LSOA name  35672 non-null  str  
dtypes: str(2)
memory usage: 1.4 MB


In [7]:
# Get all the crime types
crime_types = sorted(df_all_street["Crime type"].dropna().unique())
print(crime_types)

# Combine LSOA's and crime_types
df_historical_analysis = df_all_lsoas.merge(
    pd.DataFrame({"Crime type": crime_types}),
    how="cross"
)

df_historical_analysis.head()

['Anti-social behaviour', 'Bicycle theft', 'Burglary', 'Criminal damage and arson', 'Drugs', 'Other crime', 'Other theft', 'Possession of weapons', 'Public order', 'Robbery', 'Shoplifting', 'Theft from the person', 'Vehicle crime', 'Violence and sexual offences']


,LSOA code,LSOA name,Crime type
0,E01000001,City of London 001A,Anti-social behaviour
1,E01000001,City of London 001A,Bicycle theft
2,E01000001,City of London 001A,Burglary
3,E01000001,City of London 001A,Criminal damage and arson
4,E01000001,City of London 001A,Drugs


In [8]:
# Find latest month in the dataset
df_all_street["Month"] = pd.to_datetime(df_all_street["Month"])

latest_month = df_all_street["Month"].max()

print("Latest month:", latest_month)

# Make df with the count of last month
last_month_counts = (
    df_all_street[df_all_street["Month"] == latest_month]
    .groupby(["LSOA code", "LSOA name", "Crime type"])
    .size()
    .reset_index(name="last_month_count")
)
last_month_counts.head()

Latest month: 2026-03-01 00:00:00


,LSOA code,LSOA name,Crime type,last_month_count
0,E01000001,City of London 001A,Bicycle theft,3
1,E01000001,City of London 001A,Other theft,3
2,E01000001,City of London 001A,Public order,2
3,E01000001,City of London 001A,Theft from the person,6
4,E01000001,City of London 001A,Vehicle crime,1


In [9]:
# Add last_mont_count to the df_historical_analysis
df_historical_analysis = df_historical_analysis.merge(
    last_month_counts,
    on=["LSOA code", "LSOA name", "Crime type"],
    how="left"
)

df_historical_analysis.head()

,LSOA code,LSOA name,Crime type,last_month_count
0,E01000001,City of London 001A,Anti-social behaviour,NaN
1,E01000001,City of London 001A,Bicycle theft,3.0
2,E01000001,City of London 001A,Burglary,NaN
3,E01000001,City of London 001A,Criminal damage and arson,NaN
4,E01000001,City of London 001A,Drugs,NaN


In [10]:
# Create df with data of last year
one_year_ago = latest_month - pd.DateOffset(months=12)

df_last_year = df_all_street[
    df_all_street["Month"] > one_year_ago
]

# Create df with last year's average
last_year_avg = (
    df_last_year
    .groupby(["LSOA code", "LSOA name", "Crime type"])
    .size()
    .reset_index(name="last_year_total")
)

months_in_window = df_last_year["Month"].nunique()

last_year_avg["last_year_monthly_avg"] = (
    last_year_avg["last_year_total"] / months_in_window
)

# Add last_year_monthly_avg to df_historical_analysis
df_historical_analysis = df_historical_analysis.drop(
    columns=["last_year_monthly_avg"],
    errors="ignore"
)

df_historical_analysis = df_historical_analysis.merge(
    last_year_avg[
        ["LSOA code", "LSOA name", "Crime type", "last_year_monthly_avg"]
    ],
    on=["LSOA code", "LSOA name", "Crime type"],
    how="left"
)

In [11]:
df_historical_analysis.info()
df_historical_analysis.head()
# Df still contains all LSOA's since we have 499408 entries (14*35.672)

<class 'pandas.DataFrame'>
RangeIndex: 499408 entries, 0 to 499407
Data columns (total 5 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   LSOA code              499408 non-null  str    
 1   LSOA name              499408 non-null  str    
 2   Crime type             499408 non-null  str    
 3   last_month_count       162572 non-null  float64
 4   last_year_monthly_avg  371397 non-null  float64
dtypes: float64(2), str(3)
memory usage: 38.1 MB


,LSOA code,LSOA name,Crime type,last_month_count,last_year_monthly_avg
0,E01000001,City of London 001A,Anti-social behaviour,NaN,0.583333
1,E01000001,City of London 001A,Bicycle theft,3.0,1.833333
2,E01000001,City of London 001A,Burglary,NaN,0.583333
3,E01000001,City of London 001A,Criminal damage and arson,NaN,0.750000
4,E01000001,City of London 001A,Drugs,NaN,0.750000


In [12]:
# create df that tracks number of crimes for each crime type for a certain month
monthly_counts = (
    df_all_street
    .groupby(["LSOA code", "LSOA name", "Crime type", "Month"])
    .size()
    .reset_index(name="monthly_count")
)

# Take the maximum
max_monthly_counts = (
    monthly_counts
    .groupby(["LSOA code", "LSOA name", "Crime type"])
    ["monthly_count"]
    .max()
    .reset_index(name="max_monthly_crime_count")
)

# Add maximum crime type to df_historical_analysis
df_historical_analysis = df_historical_analysis.drop(
    columns=["max_monthly_crime_count"],
    errors="ignore"
)

df_historical_analysis = df_historical_analysis.merge(
    max_monthly_counts,
    on=["LSOA code", "LSOA name", "Crime type"],
    how="left"
)
df_historical_analysis.head()

,LSOA code,LSOA name,Crime type,last_month_count,last_year_monthly_avg,max_monthly_crime_count
0,E01000001,City of London 001A,Anti-social behaviour,NaN,0.583333,3.0
1,E01000001,City of London 001A,Bicycle theft,3.0,1.833333,7.0
2,E01000001,City of London 001A,Burglary,NaN,0.583333,2.0
3,E01000001,City of London 001A,Criminal damage and arson,NaN,0.750000,4.0
4,E01000001,City of London 001A,Drugs,NaN,0.750000,3.0


In [13]:
# Get the total crime per LSOA for the last month
lsoa_totals = (
    df_historical_analysis
    .groupby(["LSOA code", "LSOA name"])["last_month_count"]
    .sum()
    .reset_index(name="lsoa_total_last_month")
)

# Add the total_lsoa to df_historical_analysis
df_historical_analysis = df_historical_analysis.merge(
    lsoa_totals,
    on=["LSOA code", "LSOA name"],
    how="left"
)

# Compute percentage of each crime type of the last month, add it to df_historical_analysis
df_historical_analysis["last_month_percentage"] = (
    df_historical_analysis["last_month_count"] /
    df_historical_analysis["lsoa_total_last_month"]
) * 100

df_historical_analysis.head()

,LSOA code,LSOA name,Crime type,last_month_count,last_year_monthly_avg,max_monthly_crime_count,lsoa_total_last_month,last_month_percentage
0,E01000001,City of London 001A,Anti-social behaviour,NaN,0.583333,3.0,16.0,NaN
1,E01000001,City of London 001A,Bicycle theft,3.0,1.833333,7.0,16.0,18.75
2,E01000001,City of London 001A,Burglary,NaN,0.583333,2.0,16.0,NaN
3,E01000001,City of London 001A,Criminal damage and arson,NaN,0.750000,4.0,16.0,NaN
4,E01000001,City of London 001A,Drugs,NaN,0.750000,3.0,16.0,NaN


In [14]:
# Count crimes for each LSOA and crime type for the last year
last_year_counts = (
    df_last_year
    .groupby(["LSOA code", "LSOA name", "Crime type"])
    .size()
    .reset_index(name="last_year_count")
)

# Count total crimes per LSOA of last year
lsoa_year_totals = (
    last_year_counts
    .groupby(["LSOA code", "LSOA name"])["last_year_count"]
    .sum()
    .reset_index(name="lsoa_total_last_year")
)

# Combine both
last_year_counts = last_year_counts.merge(
    lsoa_year_totals,
    on=["LSOA code", "LSOA name"],
    how="left"
)

# Compute the percentage for each crime type and LSOA of the total crimes that year
last_year_counts["last_year_percentage"] = (
    last_year_counts["last_year_count"] /
    last_year_counts["lsoa_total_last_year"]
) * 100

# Add last_year_percentage to df_historical_analysis
df_historical_analysis = df_historical_analysis.drop(
    columns=["last_year_percentage"],
    errors="ignore"
)

df_historical_analysis = df_historical_analysis.merge(
    last_year_counts[
        ["LSOA code", "LSOA name", "Crime type", "last_year_percentage"]
    ],
    on=["LSOA code", "LSOA name", "Crime type"],
    how="left"
)

df_historical_analysis.head()

,LSOA code,LSOA name,Crime type,last_month_count,last_year_monthly_avg,max_monthly_crime_count,lsoa_total_last_month,last_month_percentage,last_year_percentage
0,E01000001,City of London 001A,Anti-social behaviour,NaN,0.583333,3.0,16.0,NaN,3.553299
1,E01000001,City of London 001A,Bicycle theft,3.0,1.833333,7.0,16.0,18.75,11.167513
2,E01000001,City of London 001A,Burglary,NaN,0.583333,2.0,16.0,NaN,3.553299
3,E01000001,City of London 001A,Criminal damage and arson,NaN,0.750000,4.0,16.0,NaN,4.568528
4,E01000001,City of London 001A,Drugs,NaN,0.750000,3.0,16.0,NaN,4.568528


In [15]:
if lsoa_locations.crs != forces.crs:
    forces = forces.to_crs(lsoa_locations.crs)

lsoa_locations_clean = lsoa_locations.copy()
forces_clean = forces.copy()

lsoa_locations_clean["geometry"] = lsoa_locations_clean["geometry"].buffer(0)
forces_clean["geometry"] = forces_clean["geometry"].buffer(0)

lsoa_points = lsoa_locations_clean[["LSOA21CD", "geometry"]].copy()
lsoa_points["geometry"] = lsoa_points.geometry.representative_point()

mapping_raw = gpd.sjoin(
    lsoa_points,
    forces_clean[["force", "geometry"]],
    how="left",
    predicate="within"
)

# Add one police force to each LSOA, if there are duplicates, keep the first one
# Some LSOA's fall in multiple police forces, but for mapping and the dashboard we have to choose one
# Because you don't want both police forces to send more police officers
# Also in the original data from uk police one LSOA had multiple forces

lsoa_force = (
    mapping_raw[["LSOA21CD", "force"]]
    .dropna()
    .sort_values("force") 
    .drop_duplicates(subset=["LSOA21CD"], keep="first")
    .rename(columns={
        "LSOA21CD": "LSOA code",
        "force": "police_force"
    })
)

# Check if there are LSOA's without a police force assigned
missing_forces = lsoa_force["police_force"].isna().sum()
print("LSOA's without force assigned:", missing_forces)

# Check if there are LSOA's that belong to multiple police forces
lsoa_force.groupby("LSOA code")["police_force"].nunique().max()

LSOA's without force assigned: 0


np.int64(1)

In [16]:
# Add a column police force to df_historical_analysis so we know in which force area the LSOA is located
df_historical_analysis = df_historical_analysis.merge(
    lsoa_force,
    on="LSOA code",
    how="left"
)

df_historical_analysis.head()

,LSOA code,LSOA name,Crime type,last_month_count,last_year_monthly_avg,max_monthly_crime_count,lsoa_total_last_month,last_month_percentage,last_year_percentage,police_force
0,E01000001,City of London 001A,Anti-social behaviour,NaN,0.583333,3.0,16.0,NaN,3.553299,city-of-london
1,E01000001,City of London 001A,Bicycle theft,3.0,1.833333,7.0,16.0,18.75,11.167513,city-of-london
2,E01000001,City of London 001A,Burglary,NaN,0.583333,2.0,16.0,NaN,3.553299,city-of-london
3,E01000001,City of London 001A,Criminal damage and arson,NaN,0.750000,4.0,16.0,NaN,4.568528,city-of-london
4,E01000001,City of London 001A,Drugs,NaN,0.750000,3.0,16.0,NaN,4.568528,city-of-london


In [17]:
# Read the (non)hotspot_dictionary.json
with open("../(non)hotspot_dictionary.json", "r") as f:
    hotspot_dict = json.load(f)

# Get a set of all LSOA codes that are hotspots
hotspot_lsoas = set()

for force, data in hotspot_dict.items():
    hotspot_lsoas.update(data.get("hotspots", []))

# Add attribute hotspot_status to df_historical_analysis
df_historical_analysis["hotspot_status"] = (
    df_historical_analysis["LSOA code"].isin(hotspot_lsoas)
)

df_historical_analysis.head()
df_historical_analysis.info()

<class 'pandas.DataFrame'>
RangeIndex: 499408 entries, 0 to 499407
Data columns (total 11 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   LSOA code                499408 non-null  str    
 1   LSOA name                499408 non-null  str    
 2   Crime type               499408 non-null  str    
 3   last_month_count         162572 non-null  float64
 4   last_year_monthly_avg    371397 non-null  float64
 5   max_monthly_crime_count  424645 non-null  float64
 6   lsoa_total_last_month    499408 non-null  float64
 7   last_month_percentage    162572 non-null  float64
 8   last_year_percentage     371397 non-null  float64
 9   police_force             499366 non-null  str    
 10  hotspot_status           499408 non-null  bool   
dtypes: bool(1), float64(6), str(4)
memory usage: 63.1 MB


In [18]:
# Check which 3 LSOA's don't have a police_force (449.408-449.366=42 42/14=3)
missing = df_historical_analysis[df_historical_analysis["police_force"].isna()]
print(missing["LSOA code"].unique()[:])

# E01015271 -> Devon and Cornwall police
# E01019000 -> Devon and Cornwall police
# E01024478 -> Kent Police

# Manually add them based on the falls within count
print(df_all_street.loc[
    df_all_street["LSOA code"] == "E01015271",
    "Falls within"
].value_counts())

print(df_all_street.loc[
    df_all_street["LSOA code"] == "E01019000",
    "Falls within"
].value_counts())

print(df_all_street.loc[
    df_all_street["LSOA code"] == "E01024478",
    "Falls within"
].value_counts())

<ArrowStringArray>
['E01015271', 'E01019000', 'E01024478']
Length: 3, dtype: str
Falls within
Devon & Cornwall Police    120
Name: count, dtype: int64
Falls within
Devon & Cornwall Police           505
Avon and Somerset Constabulary      1
Name: count, dtype: int64
Falls within
Kent Police                    302
Metropolitan Police Service      7
Name: count, dtype: int64


In [19]:
# Manual corrections
df_historical_analysis.loc[
    df_historical_analysis["LSOA code"] == "E01015271",
    "police_force"
] = "devon-and-cornwall"

df_historical_analysis.loc[
    df_historical_analysis["LSOA code"] == "E01019000",
    "police_force"
] = "devon-and-cornwall"

df_historical_analysis.loc[
    df_historical_analysis["LSOA code"] == "E01024478",
    "police_force"
] = "kent" 

In [20]:
# Calculate total number of recorded crimes among the entire dataset
all_time_counts = (
    df_all_street
    .groupby(["LSOA code", "LSOA name", "Crime type"])
    .size()
    .reset_index(name="all_time_total")
)

# Number of months in the dataset
total_months = df_all_street["Month"].nunique()

all_time_counts["all_time_monthly_avg"] = (
    all_time_counts["all_time_total"] / total_months
)

# Add attribute for average crimecount of the entire dataset
df_historical_analysis = df_historical_analysis.drop(
    columns=["all_time_monthly_avg"],
    errors="ignore"
)

df_historical_analysis = df_historical_analysis.merge(
    all_time_counts[
        ["LSOA code", "LSOA name", "Crime type", "all_time_monthly_avg"]
    ],
    on=["LSOA code", "LSOA name", "Crime type"],
    how="left"
)

In [21]:
# Set Nan values to 0
df_historical_analysis = df_historical_analysis.fillna(0)

# Convert df_historical_analysis to csv file
df_historical_analysis.to_csv("../data/df_historical_analysis.csv", index=False)